# Phase 5 — Baseline training: EfficientNet-B0, ANA Positive vs Negative

**Before running:** `Runtime → Change runtime type → T4 GPU`, and make sure
`My Drive/ANA_PosNeg/data/AIDA_processed.zip` exists.

**What this notebook does, per cross-validation round *k*:**
1. Split data into train / validation / test by patient (`get_round`, same code as `scripts/splits.py`).
2. Fine-tune an ImageNet-pretrained EfficientNet-B0 on 512×512 crops.
3. Keep the epoch with the best **validation balanced accuracy** (early stopping).
4. Save that model + its raw outputs (**logits**) for validation and test images to Drive.

Step 4 means Phase 7 (calibration) and Phase 8 (evaluation) can reuse these outputs **without retraining**.

**Plan:** first run only round 0 (`ROUNDS = [0]`) and inspect the curves. If they look healthy, set `ROUNDS = [0, 1, 2, 3, 4]` and run again — finished rounds are skipped automatically.

## 1. Get the data onto Colab's local disk

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil, time, zipfile
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/ANA_PosNeg")
ZIP_ON_DRIVE = DRIVE_ROOT / "data" / "AIDA_processed.zip"
DATA = Path("/content/data")          # Colab's local disk: fast, but wiped when the session ends

# Reading 2,073 small files straight from Drive is slow; one zip copy + unzip is fast.
if not (DATA / "manifest.csv").exists():
    t = time.time()
    shutil.copy(ZIP_ON_DRIVE, "/content/AIDA_processed.zip")
    with zipfile.ZipFile("/content/AIDA_processed.zip") as zf:
        zf.extractall(DATA)
    print(f"Copied + unzipped in {time.time() - t:.0f} s")
print("PNG images:", len(list((DATA / "processed").rglob("*.png"))))

In [ ]:
!pip install -q timm

## 2. Settings

| Setting | Value | Meaning / why |
|---|---|---|
| `model_name` | efficientnet_b0 | small, fast pretrained CNN (baseline) |
| `img_size` | 512 | square crop side (~75% of each image, full resolution) |
| `batch_size` | 32 | images per weight update; fits a T4 with mixed precision |
| `epochs` | 30 | maximum passes over the training set |
| `lr` | 3e-4 | learning rate: size of each weight update step |
| `weight_decay` | 1e-2 | gently pulls weights toward zero → less overfitting |
| `warmup_epochs` | 1 | start with tiny steps so the pretrained weights aren't wrecked at the start |
| `patience` | 7 | stop if validation hasn't improved for 7 epochs |
| `brightness` | 0.1 | brightness randomly ×0.9–1.1 only (brightness is the signal!) |

In [ ]:
CFG = dict(
    model_name="efficientnet_b0",
    img_size=512,
    batch_size=32,
    epochs=30,
    lr=3e-4,
    weight_decay=1e-2,
    warmup_epochs=1,
    patience=7,
    brightness=0.1,
    threshold=0.5,
    seed=42,
    num_workers=2,
)
ROUNDS = [0]          # start with round 0 only; later: [0, 1, 2, 3, 4]

OUT = DRIVE_ROOT / "outputs" / CFG["model_name"]
OUT.mkdir(parents=True, exist_ok=True)

## 3. Imports, reproducibility, and the train/val/test split (identical to `scripts/splits.py`)

In [ ]:
import json, math, random
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
import warnings
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

# Harmless: with mixed precision, GradScaler may skip the very first optimizer step
# while it calibrates, which makes PyTorch print a misleading scheduler-order warning.
warnings.filterwarnings("ignore", message=r"Detected call of `lr_scheduler\.step\(\)` before")

device = "cuda"
torch.backends.cudnn.benchmark = True   # let cuDNN pick the fastest convolution algorithm


def seed_everything(seed):
    # Same seed -> same random choices (crops, flips, initial head weights) -> repeatable runs
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


VAL_PARTS = 6

def get_round(manifest, k, seed=42):
    # Returns (train_df, val_df, test_df) for cross-validation round k.
    # numpy's legacy RandomState gives the same sequence in every numpy version,
    # so this split is identical on the laptop and on Colab.
    test = manifest[manifest["fold"] == k]
    rest = manifest[manifest["fold"] != k]
    patients = rest.drop_duplicates("sample_id")[["sample_id", "label"]].sort_values("sample_id")
    rng = np.random.RandomState(seed + k)
    val_ids = []
    for label in (0, 1):
        ids = patients.loc[patients["label"] == label, "sample_id"].to_numpy()
        ids = ids[rng.permutation(len(ids))]
        val_ids.extend(ids[: round(len(ids) / VAL_PARTS)])
    is_val = rest["sample_id"].isin(val_ids)
    return rest[~is_val], rest[is_val], test


manifest = pd.read_csv(DATA / "manifest.csv")
# These numbers must match outputs/phase4_folds_log.txt from the laptop.
for k in range(5):
    tr, va, te = get_round(manifest, k, CFG["seed"])
    print(f"round {k}: train {len(tr)}  val {len(va)}  test {len(te)}")

## 4. Load all images into memory once

2,073 images × ~1 MB = ~2.2 GB of RAM (Colab has ~12 GB). Decoding PNGs once instead of every epoch makes training several times faster.

In [ ]:
def load_png(rel_path):
    img = Image.open(DATA / "processed" / rel_path).convert("RGB")
    return torch.from_numpy(np.asarray(img)).permute(2, 0, 1).contiguous()   # HWC -> CHW, uint8

t = time.time()
with ThreadPoolExecutor(8) as ex:
    IMAGES = dict(zip(manifest["image_path"], ex.map(load_png, manifest["image_path"])))
gb = sum(x.numel() for x in IMAGES.values()) / 1e9
print(f"Loaded {len(IMAGES)} images ({gb:.1f} GB) in {time.time() - t:.0f} s")

## 5. Augmentation and the Dataset

**Training** (random, different every epoch): 512×512 random crop → horizontal/vertical flip → random 90° rotation → mild brightness change.
Cells have no natural "up", so rotated/flipped cells are still valid examples.

**Validation / test** (fixed): 512×512 **centre** crop only — the same image always gives the same answer.

Both then: pixel values 0–255 → 0–1 → **normalised with ImageNet mean/std** (the pretrained network expects inputs on that scale).

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class RandomRot90(nn.Module):
    def forward(self, x):
        return torch.rot90(x, int(torch.randint(0, 4, (1,))), dims=(-2, -1))


train_tf = v2.Compose([
    v2.RandomCrop(CFG["img_size"]),
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    RandomRot90(),
    v2.ColorJitter(brightness=CFG["brightness"]),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])
eval_tf = v2.Compose([
    v2.CenterCrop(CFG["img_size"]),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])


class HEp2Dataset(Dataset):
    # Gives (image tensor, label) pairs. The model never sees sample_id -- only pixels.
    def __init__(self, df, transform):
        self.paths = df["image_path"].tolist()
        self.labels = df["label"].astype("float32").tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(IMAGES[self.paths[i]]), torch.tensor(self.labels[i])


def make_loader(df, transform, train):
    return DataLoader(HEp2Dataset(df, transform), batch_size=CFG["batch_size"], shuffle=train,
                      drop_last=train, num_workers=CFG["num_workers"], pin_memory=True,
                      persistent_workers=CFG["num_workers"] > 0)


# Quick visual check: what the model actually receives during training
tr, _, _ = get_round(manifest, 0, CFG["seed"])
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
ds = HEp2Dataset(tr.sample(6, random_state=1), train_tf)
for ax, (x, y) in zip(axes, (ds[i] for i in range(6))):
    img = (x.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)
    ax.imshow(img); ax.set_title("Positive" if y == 1 else "Negative"); ax.axis("off")
plt.show()

## 6. Training helpers

- **Mixed precision (`autocast` + `GradScaler`)**: most maths in 16-bit instead of 32-bit → ~2× faster on a T4, same accuracy. `GradScaler` prevents tiny gradients from rounding to zero in 16-bit.
- **Loss = `BCEWithLogitsLoss(pos_weight = N_neg / N_pos)`**: binary cross-entropy on the raw output. `pos_weight ≈ 0.39` shrinks each positive image's loss so that positives and negatives count **equally in total** (fixes the 2.6 : 1 imbalance).
- **Learning-rate schedule**: 1 epoch of warm-up (LR rises from 0), then **cosine decay** (LR smoothly falls to 0) — big steps early, careful fine steps late.
- **Metrics**: balanced accuracy (at threshold 0.5), sensitivity, specificity, ROC-AUC.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def compute_metrics(labels, logits, threshold=CFG["threshold"]):
    labels = np.asarray(labels).astype(int)
    prob = sigmoid(np.asarray(logits))
    pred = (prob >= threshold).astype(int)
    sens = ((pred == 1) & (labels == 1)).sum() / (labels == 1).sum()   # positives caught
    spec = ((pred == 0) & (labels == 0)).sum() / (labels == 0).sum()   # negatives cleared
    return dict(bal_acc=(sens + spec) / 2, sensitivity=sens, specificity=spec,
                auc=roc_auc_score(labels, prob))


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = criterion(model(x).squeeze(1), y)      # forward pass + loss
        scaler.scale(loss).backward()                      # backward pass: compute gradients
        scaler.step(optimizer)                             # update weights
        scaler.update()
        scheduler.step()                                   # update learning rate
        total += loss.item() * len(y); n += len(y)
    return total / n


@torch.no_grad()                                           # no gradients needed when only predicting
def predict_logits(model, loader):
    model.eval()                                           # eval mode: BatchNorm/dropout behave deterministically
    out = []
    for x, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(model(x.to(device, non_blocking=True)).squeeze(1).float().cpu())
    return torch.cat(out).numpy()

## 7. One cross-validation round

Early stopping keeps the epoch with the highest **validation balanced accuracy** (ties → lower validation loss).
The best model and a `predictions.csv` (raw logits for every validation and test image) are saved to Drive.

In [ ]:
def train_round(k):
    rdir = OUT / f"round_{k}"
    rdir.mkdir(parents=True, exist_ok=True)
    if (rdir / "predictions.csv").exists():
        print(f"Round {k} already finished -> skipped"); return

    seed_everything(CFG["seed"] + k)
    train_df, val_df, test_df = get_round(manifest, k, CFG["seed"])
    train_loader = make_loader(train_df, train_tf, train=True)
    val_loader = make_loader(val_df, eval_tf, train=False)
    test_loader = make_loader(test_df, eval_tf, train=False)

    # Pretrained backbone; its 1000-class ImageNet head is replaced by 1 output (the logit).
    model = timm.create_model(CFG["model_name"], pretrained=True, num_classes=1).to(device)

    n_pos = int(train_df["label"].sum()); n_neg = len(train_df) - n_pos
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(n_neg / n_pos, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = CFG["epochs"] * len(train_loader)
    warm = CFG["warmup_epochs"] * len(train_loader)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (steps - warm))))
    scaler = torch.amp.GradScaler()

    val_y = torch.tensor(val_df["label"].values, dtype=torch.float32, device=device)
    print(f"Round {k}: train {len(train_df)} (P {n_pos}/N {n_neg}), val {len(val_df)}, test {len(test_df)}, "
          f"pos_weight {n_neg / n_pos:.2f}")

    history, best, bad = [], (-1.0, float("inf")), 0
    for epoch in range(1, CFG["epochs"] + 1):
        t = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler)
        val_logits = predict_logits(model, val_loader)
        val_loss = criterion(torch.tensor(val_logits, device=device), val_y).item()
        m = compute_metrics(val_df["label"].values, val_logits)
        history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss, **m))

        improved = (m["bal_acc"] > best[0]) or (m["bal_acc"] == best[0] and val_loss < best[1])
        if improved:
            best, bad = (m["bal_acc"], val_loss), 0
            torch.save(model.state_dict(), rdir / "best.pt")
        else:
            bad += 1
        print(f"  ep {epoch:2d} | train loss {train_loss:.3f} | val loss {val_loss:.3f} | "
              f"bal_acc {m['bal_acc']:.3f} sens {m['sensitivity']:.3f} spec {m['specificity']:.3f} "
              f"auc {m['auc']:.3f} | {time.time() - t:.0f}s {'*best' if improved else ''}")
        if bad >= CFG["patience"]:
            print(f"  early stop: no improvement for {CFG['patience']} epochs"); break

    pd.DataFrame(history).to_csv(rdir / "history.csv", index=False)

    # Reload the best epoch and save raw outputs for validation + test images
    model.load_state_dict(torch.load(rdir / "best.pt"))
    parts = []
    for name, df, loader in (("val", val_df, val_loader), ("test", test_df, test_loader)):
        parts.append(df.assign(split=name, logit=predict_logits(model, loader))
                       [["image_path", "sample_id", "label", "split", "logit"]])
    preds = pd.concat(parts)
    preds.to_csv(rdir / "predictions.csv", index=False)

    test = preds[preds["split"] == "test"]
    summary = dict(round=k, best_epoch=int(pd.DataFrame(history)["bal_acc"].idxmax() + 1),
                   **{f"test_{a}": float(b) for a, b in compute_metrics(test["label"], test["logit"]).items()})
    (rdir / "summary.json").write_text(json.dumps(summary, indent=2))
    print("  TEST (uncalibrated, threshold 0.5):", {a: round(b, 3) for a, b in summary.items()})


(OUT / "config.json").write_text(json.dumps(CFG, indent=2))
for k in ROUNDS:
    train_round(k)

## 8. Learning curves

How to read them:
- **Both losses falling** → the model is learning.
- **Train loss falling but validation loss rising** → **overfitting** (memorising training images).
- **Validation balanced accuracy** should rise and then flatten; the `*best` epoch is the one saved.

In [ ]:
BLUE, ORANGE = "#2a78d6", "#eb6834"
for rdir in sorted(OUT.glob("round_*")):
    if not (rdir / "history.csv").exists():
        continue
    h = pd.read_csv(rdir / "history.csv")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.3))
    a1.plot(h.epoch, h.train_loss, color=BLUE, lw=2, label="train")
    a1.plot(h.epoch, h.val_loss, color=ORANGE, lw=2, label="validation")
    a1.set_title(f"{rdir.name}: loss", loc="left"); a1.set_xlabel("epoch"); a1.legend(frameon=False)
    a2.plot(h.epoch, h.bal_acc, color=ORANGE, lw=2, label="balanced accuracy")
    a2.plot(h.epoch, h.auc, color=BLUE, lw=2, label="AUC")
    a2.set_title(f"{rdir.name}: validation metrics", loc="left"); a2.set_xlabel("epoch"); a2.legend(frameon=False)
    for a in (a1, a2):
        a.spines[["top", "right"]].set_visible(False); a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## 9. Summary of finished rounds (preliminary)

Test results here are **uncalibrated** and at threshold 0.5. Calibration (Phase 7) and the full evaluation (Phase 8) come later.

In [ ]:
rows = [json.loads(p.read_text()) for p in sorted(OUT.glob("round_*/summary.json"))]
if rows:
    res = pd.DataFrame(rows).set_index("round")
    display(res.round(3))
    if len(res) > 1:
        print("mean ± std:")
        for c in [c for c in res.columns if c.startswith("test_")]:
            print(f"  {c:18s} {res[c].mean():.3f} ± {res[c].std():.3f}")